## Group Debrief: Model Comparison (Week 4)

### A note on comparability before you start

Unlike Week 3 (where all 4-5 exercises predicted the same target on the same dataset and could be lined up in one big table), Week 4's exercises span **two genuinely different problem types**:

- **Exercises 1-2 (K-Means, DBSCAN)** are **unsupervised clustering** on the Mall Customer dataset -> there is no "correct answer" and no prediction error, so they are evaluated with cluster-quality metrics (Silhouette, Davies-Bouldin).
- **Exercises 3-5 (Baseline, ARIMA/SARIMA, ML Forecasting)** are **supervised time-series forecasting** on the AirPassengers dataset -> they predict a known future value, so they are evaluated with prediction-error metrics (MAE, RMSE, MAPE).

**You cannot put a Silhouette Score next to an RMSE and ask "which is better."** They measure different things, on different datasets, for different tasks. Instead:
- Compare **within** each subgroup numerically (Section 1 and Section 2 below).
- Compare **across** subgroups only qualitatively -> workflow, interpretability, effort, and business usefulness (Section 3).

This mirrors how Week 3 flagged Multiple Linear Regression as "not directly comparable" on classification metrics -> here, the same idea applies to an entire half of the group's exercises.

---

### 1. Clustering Comparison (Exercises 1-2)

| Exercise | Model | Chosen Hyperparameter(s) | # Clusters Found | Inertia (SSE) | Silhouette Score | Davies-Bouldin Index |
|---|---|---|---|---|---|---|
| K-Means | KMeans | k = 6 | 6| 134.35| 0.431| 0.835| |
| DBSCAN | DBSCAN | eps =0.5, min_samples = 5| 6 | n/a | 0.482 |0.639 |

> *Note: DBSCAN's Silhouette/Davies-Bouldin exclude noise points; K-Means' do not. Keep this in mind when comparing the two rows.*

---

### 2. Forecasting Comparison (Exercises 3-5, all on the same 24-month AirPassengers test set)

| Exercise | Model(s) | Key Hyperparameter(s) Tuned | MAE | RMSE | MAPE | R² / AIC |
|---|---|---|---|---|---|---|
| Baseline | Naive | - |115.25 |137.33 |23.58 | n/a |
| Baseline | Seasonal Naive | - |47.58 | 49.99| 10.52| n/a |
| Baseline | Moving Average | window | 77.83| 103.21|15.47 | n/a |
| Classical | ARIMA (best order) | (p,d,q) |69.10 |90.70 |13.78 | AIC = 1093.07|
| Classical | SARIMA (best order) | (p,d,q)(P,D,Q,12) | 43.53| 47.26|9.57 | AIC = 806.87|
| ML | Random Forest | n_estimators=400, max_depth=6 | 32.84 | 46.50 | 6.65 | R² = 0.612 |
| ML | XGBoost | learning_rate=0.1, n_estimators=200 | 37.55 | 50.09 | 7.66 | R² = 0.550 |

---

### 3. Cross-Model Discussion Points

| Question | Notes |
|---|---|
| Which clustering method (K-Means vs. DBSCAN) produced more interpretable, business-usable customer segments? | DBSCAN |
| Did K-Means and DBSCAN roughly agree on how many meaningful customer groups exist? | Yes, at 6 |
| Which forecasting method had the lowest RMSE on the test set? | Random Forest (RMSE = 46.50), narrowly ahead of SARIMA (47.26) |
| Did the classical (ARIMA/SARIMA) or ML (RF/XGBoost) models actually beat the simple baselines -> by how much? | Yes. The best baseline was Seasonal Naive (RMSE 49.99). SARIMA (47.26) and Random Forest (46.50) beat it, cutting RMSE by ~5-7%. ARIMA (90.70) and XGBoost (50.09) did NOT clearly beat Seasonal Naive, showing that added complexity only pays off when it captures the yearly seasonality. |
| Which model was easiest to explain to a non-ML person -> across *all 5* exercises? | The Baseline forecasts (especially Seasonal Naive: "next July = last July"). Among the advanced models, K-Means clusters are the most intuitive to explain visually. |
| Which required the most manual feature engineering / preprocessing effort? | The ML models (Random Forest / XGBoost) -> they needed hand-built lag, rolling-mean, and calendar features to turn the time series into a tabular dataset, unlike ARIMA/SARIMA which model the series directly. |
| Which visualization (across all 5 exercises) was most convincing for explaining *why* a model performed the way it did? | The decomposition plot (Ex 3) for showing the strong yearly seasonality that made Seasonal Naive/SARIMA work, and the feature-importance charts (Ex 5) showing lag_12 dominating -> both point to the same yearly pattern. |
| If the mall wanted **both** customer segments *and* a passenger-demand forecast, which combination of two models (one from Ex. 1-2, one from Ex. 3-5) would you recommend piecing together, and why? | **K-Means (segments) + SARIMA (forecast).** K-Means gives clean, business-usable customer groups that are easy to explain to stakeholders, and SARIMA gives a strong, interpretable seasonal forecast that beat the baseline while being less fiddly than the feature-engineered ML models. Together they cover "who are our customers" and "what is future demand" with the best balance of accuracy and explainability. |

---

### 4. Hyperparameter Effects Summary

| Exercise | Hyperparameter(s) | Effect on Performance |
|---|---|---|
| K-Means | k (elbow / silhouette) | At k = 6, silhouette score = 0.431 |
| DBSCAN | eps, min_samples | at eps = 0.5, silhouette score = 0.482|
| Baseline | window (moving average) | A 12-month window gave a flat forecast (RMSE 103.21) -> better than plain Naive but far worse than Seasonal Naive, since a flat line cannot follow the seasonal peaks. |
| ARIMA / SARIMA | (p,d,q), seasonal order | Adding the seasonal order (SARIMA) massively improved the fit -> AIC dropped from 1093 (ARIMA) to 807 (SARIMA) and RMSE from 90.70 to 47.26, because it captures the yearly cycle ARIMA ignores. |
| Random Forest | n_estimators, max_depth | Raising n_estimators 100 -> 200 -> 400 gave a small, steady RMSE drop (47.87 -> 46.95 -> 46.50); best at 400 trees with max_depth=6. |
| XGBoost | learning_rate, n_estimators | learning_rate mattered most: 0.01 was far too slow (RMSE 103.7), 0.1 was best (50.09), and 0.3 overshot (56.26). |

---

### 5. Final Group Rankings (within each subgroup only -- do NOT rank across subgroups)

**Clustering (by Silhouette Score, higher is better):**

| Rank | Exercise | Model | Silhouette Score |
|---|---|---|---|
| 1 | 02 | DBSCAN | 0.482 |
| 2 | 01 |K-Means | 0.431 |

**Forecasting (by test-set RMSE, lower is better):**

| Rank | Exercise | Model | RMSE |
|---|---|---|---|
| 1 | 05 | Random Forest | 46.50 |
| 2 | 04 | SARIMA | 47.26 |
| 3 | 03 | Seasonal Naive | 49.99 |
| 4 | 05 | XGBoost | 50.09 |
| 5 | 04 | ARIMA | 90.70 |
| 6 | 03 | Moving Average | 103.21 |
| 7 | 03 | Naive | 137.33 |

---

### 6. Group Summary Paragraph

*(5 sentences, written together as a group wrapping up Week 4's key takeaways -> touch on both clustering and forecasting, and on why they aren't directly comparable)*

> In Week 4 we tackled two very different kinds of problems: unsupervised clustering of mall customers (K-Means, DBSCAN) and supervised time-series forecasting of airline passengers (Baseline, ARIMA/SARIMA, ML). For clustering, DBSCAN produced slightly better-separated segments (Silhouette 0.482 vs. 0.431 for K-Means), and both methods agreed on roughly six meaningful customer groups. For forecasting, the strong yearly seasonality of the AirPassengers data was the deciding factor -> the models that captured it (Random Forest with lag features, SARIMA, and even the simple Seasonal Naive) all landed near an RMSE of ~47-50, while methods that ignored seasonality (Naive, Moving Average, plain ARIMA) fell far behind. The advanced models only narrowly beat the Seasonal Naive baseline, which is an important lesson: extra complexity and feature engineering are only worthwhile when they capture something real the baseline misses. Crucially, the clustering and forecasting results cannot be ranked against each other -> a Silhouette Score and an RMSE measure completely different things on different datasets, so we compared each subgroup only within itself.

---

### 7. Open Questions for Further Exploration

- If the group had another hour, would you spend it improving a clustering model, a forecasting model, or trying the optional bonus models (Prophet, LSTM/GRU)?
- How would you combine unsupervised and supervised techniques in a single real-world pipeline (e.g., cluster customers first, then forecast demand per cluster)?
- What would change about your evaluation approach if the clustering task also had to be validated on unseen/future customers, the way the forecasting task is validated on unseen/future months?